### **Librerías**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from scipy.optimize import minimize
import warnings

# Configuración inicial
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.2f}'.format

### **Parámetros**

In [ ]:
os.chdir("/Users/juanchacon/Library/Mobile Documents/com~apple~CloudDocs/GitHub/bot_futuros")

SILVER_DATA_PATH = os.getcwd() + "/data/silver"

### **Funciones**

In [ ]:
def cargar_y_procesar(ruta_archivo):
    """Carga el dataset y establece la fecha como índice."""
    df = pd.read_csv(ruta_archivo)
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df.set_index('Fecha', inplace=True)

    # Asignar frecuencia diaria (ajustar si los datos no son diarios)
    df = df.asfreq('D')
    return df


def calcular_metricas(y_true, y_pred, nombre_modelo, variable):
    """Calcula las métricas de error solicitadas."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred) # Error Absoluto (Promedio)
    me = np.mean(y_true - y_pred)             # Error (Sesgo Promedio)
    
    return {
        'Variable': variable,
        'Modelo': nombre_modelo,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'Error Absoluto (MAE)': mae,
        'Error (ME)': me
    }


def optimizar_ensamble(predicciones_train, y_train):
    """
    Encuentra los pesos óptimos para combinar los modelos minimizando el MSE
    en el conjunto de entrenamiento.
    """
    def loss_func(weights):
        # Combinación lineal de las predicciones
        final_pred = np.dot(predicciones_train, weights)
        return mean_squared_error(y_train, final_pred)
    
    # Restricción: La suma de los pesos debe ser 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    # Límites: Los pesos deben estar entre 0 y 1
    bounds = [(0, 1)] * predicciones_train.shape[1]
    
    # Pesos iniciales iguales
    init_weights = [1/predicciones_train.shape[1]] * predicciones_train.shape[1]
    
    # Optimización
    result = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    return result.x


def ejecutar_pronosticos(df):
    columnas_objetivo = df.columns
    consolidado = []
    
    # Dividir 80% Train, 20% Test
    n_rows = len(df)
    train_size = int(n_rows * 0.8)
    
    print(f"Total datos: {n_rows} | Train: {train_size} | Test: {n_rows - train_size}")
    
    for col in columnas_objetivo:
        print(f"\nProcesando: {col}...")
        series = df[col]
        train, test = series.iloc[:train_size], series.iloc[train_size:]
        
        preds_test = {}    # Para guardar predicciones en Test
        preds_train = {}   # Para guardar valores ajustados en Train (para el ensamble)
        
        # --- 1. Promedio Móvil (Optimizado) ---
        best_mse = float('inf')
        best_w = 3
        # Buscamos la mejor ventana entre 2 y 30 días
        for w in range(2, 31):
            rolling_train = train.rolling(window=w).mean().shift(1).dropna()
            if len(rolling_train) > 0:
                mse_val = mean_squared_error(train.loc[rolling_train.index], rolling_train)
                if mse_val < best_mse:
                    best_mse = mse_val
                    best_w = w
        
        # Pronóstico (Naive: proyectamos el último valor del promedio móvil)
        last_ma = train.rolling(window=best_w).mean().iloc[-1]
        pred_ma = pd.Series([last_ma] * len(test), index=test.index)
        preds_test['MA'] = pred_ma
        # Guardamos ajuste en train para el ensamble
        preds_train['MA'] = train.rolling(window=best_w).mean().shift(1)
        consolidado.append(calcular_metricas(test, pred_ma, f'Promedio Móvil (w={best_w})', col))

        # --- 2. Suavización Exponencial Simple (SES) ---
        ses_model = SimpleExpSmoothing(train).fit(optimized=True)
        pred_ses = ses_model.forecast(len(test))
        preds_test['SES'] = pred_ses
        preds_train['SES'] = ses_model.fittedvalues
        consolidado.append(calcular_metricas(test, pred_ses, 'SES', col))

        # --- 3. Holt-Winters ---
        # Asumimos estacionalidad semanal (period=7)
        try:
            hw_model = ExponentialSmoothing(train, seasonal_periods=7, trend='add', seasonal='add').fit()
            pred_hw = hw_model.forecast(len(test))
            preds_test['HW'] = pred_hw
            preds_train['HW'] = hw_model.fittedvalues
            consolidado.append(calcular_metricas(test, pred_hw, 'Holt-Winters', col))
        except:
            print("  Warning: Holt-Winters falló, usando versión sin estacionalidad.")
            hw_model = ExponentialSmoothing(train, trend='add').fit()
            pred_hw = hw_model.forecast(len(test))
            preds_test['HW'] = pred_hw
            preds_train['HW'] = hw_model.fittedvalues
            consolidado.append(calcular_metricas(test, pred_hw, 'Holt-Winters (Fallback)', col))

        # --- 4. ARIMA ---
        # Usamos un orden fijo (5,1,0) para velocidad, o un bucle pequeño para optimizar
        try:
            arima_model = ARIMA(train, order=(5,1,0)).fit()
            pred_arima = arima_model.forecast(len(test))
            preds_test['ARIMA'] = pred_arima
            preds_train['ARIMA'] = arima_model.fittedvalues
            consolidado.append(calcular_metricas(test, pred_arima, 'ARIMA (5,1,0)', col))
        except:
            pred_arima = pd.Series([train.mean()] * len(test), index=test.index)
            preds_test['ARIMA'] = pred_arima
            preds_train['ARIMA'] = pd.Series([train.mean()]*len(train), index=train.index)
            consolidado.append(calcular_metricas(test, pred_arima, 'ARIMA (Error)', col))

        # --- 5. Sistema Ponderado (Ensamble) ---
        # Alinear datos de entrenamiento (eliminar NaNs iniciales por lags)
        df_train_preds = pd.DataFrame(preds_train)
        df_train_preds['Actual'] = train
        df_train_preds.dropna(inplace=True)
        
        X_train = df_train_preds[['MA', 'SES', 'HW', 'ARIMA']].values
        y_train_vals = df_train_preds['Actual'].values
        
        # Obtener mejores pesos
        pesos = optimizar_ensamble(X_train, y_train_vals)
        pesos_str = ", ".join([f"{w:.2f}" for w in pesos])
        
        # Aplicar pesos al Test
        X_test = np.column_stack([preds_test['MA'], preds_test['SES'], preds_test['HW'], preds_test['ARIMA']])
        pred_ensamble = np.dot(X_test, pesos)
        pred_ensamble_series = pd.Series(pred_ensamble, index=test.index)
        
        consolidado.append(calcular_metricas(test, pred_ensamble_series, f'Ensamble (W=[{pesos_str}])', col))

    return pd.DataFrame(consolidado)

In [ ]:
archivo = 'datos_DEMANDA_COMPRADOR.csv' # Asegúrate de que el archivo esté en la misma carpeta
try:
    df_datos = cargar_y_procesar(f"{SILVER_DATA_PATH}/{archivo}")
    df_resultados = ejecutar_pronosticos(df_datos)
    
    print("\n--- CONSOLIDADO DE PRONÓSTICOS ---")
    print(df_resultados)
    
    # Guardar resultados
    #df_resultados.to_csv('resultado_pronosticos.csv', index=False)
    #print("\nResultados guardados en 'resultado_pronosticos.csv'")
    
except FileNotFoundError:
    print("Error: No se encontró el archivo CSV.")